# 🐍 Day 22 · Python Basics — Classify, Fan-out & Combine (Router and Map-Reduce Agents)

> **Goal of this notebook:** A router agent picks the right knowledge sources, sends the question to all of them *at the same time*, and combines the answers.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

✅ No API key and no internet needed for this notebook.

### 📋 What you will learn
1. Multi-label classification with keywords
2. A dictionary of knowledge sources
3. `Send`-style task objects
4. Running tasks in parallel and collecting them with `as_completed`
5. Combining the answers

## 1. Classification
A question can belong to **more than one** category (multi-label).

In [ ]:
categories = {
    "github": ["code", "bug", "repo", "pull request"],
    "notion": ["policy", "process", "docs", "onboarding"],
    "slack":  ["discussion", "team", "announced", "channel"],
}

def classify(question):
    q = question.lower()
    return [name for name, words in categories.items() if any(w in q for w in words)]

print(classify("Is there a bug in the auth repo?"))
print(classify("What was announced about the onboarding process?"))

## 2. Knowledge sources
Each source is a function. In the real notebook each one is a small agent with its own tools.

In [ ]:
import time

def search_github(q):
    time.sleep(0.5); return "GitHub: issue #42 'login bug' is open"
def search_notion(q):
    time.sleep(0.5); return "Notion: onboarding takes 2 weeks"
def search_slack(q):
    time.sleep(0.5); return "Slack: new onboarding announced in #general"

sources = {"github": search_github, "notion": search_notion, "slack": search_slack}

## 3. `Send`-style tasks
LangGraph's `Send("node", data)` means *run this node with this data*. We can represent it as a list of dictionaries.

In [ ]:
question = "What was announced in the team channel about the onboarding process?"
tasks = [{"node": src, "question": question} for src in classify(question)]
for t in tasks:
    print(t)

## 4. Run in parallel and collect as they finish

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

start = time.time()
results = {}
with ThreadPoolExecutor() as pool:
    futures = {pool.submit(sources[t["node"]], t["question"]): t["node"] for t in tasks}
    for future in as_completed(futures):          # gives each job when it is done
        results[futures[future]] = future.result()

print(f"Took {time.time() - start:.1f} s for {len(tasks)} sources")
for name, answer in results.items():
    print(f"- {answer}")

## 5. 🔗 Combine the answers

In [ ]:
def synthesize(question, results):
    lines = [f"Question: {question}", "Answer based on:"]
    lines += [f"  • {results[name]}" for name in sorted(results)]
    return "\n".join(lines)

print(synthesize(question, results))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add the word `"error"` to the `github` category and test `classify("I got an error")`.
2. What does `classify` return for a question with no keywords? How would you handle that?
3. Add a `"jira"` source and category.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
categories["github"].append("error")
print(classify("I got an error"))

print(classify("hello"))   # [] -> you could fall back to searching ALL sources
chosen = classify("hello") or list(sources)   # 'or' picks the second value when the first is empty
print(chosen)
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Multi-label | `[n for n, ws in cats.items() if any(...)]` |
| Task objects | `{'node': 'github', 'question': q}` |
| Submit job | `pool.submit(func, arg)` |
| As they finish | `for f in as_completed(futures):` |
| Job result | `future.result()` |
| Fallback | `classify(q) or list(sources)` |

## ➡️ What's next?
You now know the Python used in **Day_22_MapReduce_and_Router_Agents**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀